## Installing important libs

In [ ]:
pip install pydub


In [ ]:
pip install webvtt-py


In [ ]:
pip install torch torchaudio librosa tqdm


In [ ]:
!pip install yt-dlp


In [ ]:
!pip install youtube-transcript-api


## Data collection 

### Scraping from youtube

In [ ]:
import os
import csv
import subprocess
from googleapiclient.discovery import build
from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound, TranscriptsDisabled, NoTranscriptAvailable
import torch

In [ ]:

API_KEY = os.environ["YOUTUBE_API_KEY"]  
SEARCH_QUERIES = [
    "أخبار عربية",
    "أخبار مصرية",
    "اللغة العربية الفصحى",
    "اللهجة المصرية",
    "برامج حوارية عربية",
    "تعلم اللغة العربية",
    "نشرات الأخبار العربية",
    "برامج سياسية عربية",
    "قنوات الأخبار المصرية",
    "تقارير إخبارية بالعربية"
]
TARGET_VIDEO_COUNT = 500
VIDEO_DURATION_FILTER = "medium" 

DOWNLOAD_DIR = "videos"
WAV_DIR = "wavs"
TRANSCRIPTIONS_DIR = "transcriptions"
METADATA_CSV = "metadata.csv"

# Create directories if they don't exist
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(WAV_DIR, exist_ok=True)
os.makedirs(TRANSCRIPTIONS_DIR, exist_ok=True)

# ------------------- YOUTUBE DATA API SEARCH ------------------- #
def search_youtube_videos(api_key, queries, target_count=500, video_duration="medium"):
    youtube = build('youtube', 'v3', developerKey=api_key)
    videos = []
    query_index = 0

    while len(videos) < target_count and query_index < len(queries):
        query = queries[query_index]
        query_page_token = None
        print(f"Searching for query: {query}")

        while len(videos) < target_count:
            request = youtube.search().list(
                q=query,
                part="snippet",
                type="video",
                videoDuration=video_duration,
                maxResults=50,
                pageToken=query_page_token
            )
            response = request.execute()

            for item in response['items']:
                video_id = item['id']['videoId']
                if video_id not in videos:
                    videos.append(video_id)
                    if len(videos) >= target_count:
                        break

            query_page_token = response.get('nextPageToken')
            if not query_page_token:
                break
        query_index += 1

    return videos

# ------------------- CHECK ARABIC TRANSCRIPTS ------------------- #
def has_arabic_transcript(video_id):
    try:
        transcripts = YouTubeTranscriptApi.list_transcripts(video_id)
        for t in transcripts:
            if t.language_code.startswith('ar'):
                return True
        return False
    except (NoTranscriptFound, TranscriptsDisabled, NoTranscriptAvailable):
        return False
    except Exception:
        return False

# ------------------- DOWNLOAD VIDEO AND SUBTITLES (AUDIO ONLY) ------------------- #
def download_video_and_subs(video_id, download_dir):
    url = f"https://www.youtube.com/watch?v={video_id}"
    cmd = [
        "yt-dlp",
        url,
        "-f", "bestaudio",          # Download only the best audio
        "--write-auto-subs",        # Write automatic subtitles if available
        "--sub-lang", "ar",         # Arabic subtitles
        "--sub-format", "vtt",      # Subtitles in VTT format
        "-o", os.path.join(download_dir, f"{video_id}.%(ext)s")
    ]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    return result.returncode == 0

# ------------------- CONVERT AUDIO TO WAV ------------------- #
def convert_to_wav(input_path, output_path, sample_rate=16000):
    cmd = [
        "ffmpeg", "-y",
        "-i", input_path,
        "-ar", str(sample_rate),
        "-ac", "1",
        output_path
    ]
    subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)



In [ ]:
# ------------------- MAIN WORKFLOW ------------------- #
def main():
    # 1. Search videos with multiple queries
    print("Searching for videos...")
    all_videos = search_youtube_videos(API_KEY, SEARCH_QUERIES, TARGET_VIDEO_COUNT, VIDEO_DURATION_FILTER)
    print(f"Found {len(all_videos)} videos from search.")

    # 2. Filter by Arabic transcripts
    print("Checking for Arabic transcripts...")
    videos_with_ar = []
    for vid in all_videos:
        if has_arabic_transcript(vid):
            videos_with_ar.append(vid)

    print(f"Found {len(videos_with_ar)} videos with Arabic transcripts.")

    # 3. Download audio and subtitles
    print("Downloading audios and subtitles...")
    metadata = []
    for idx, vid in enumerate(videos_with_ar, start=1):
        print(f"({idx}/{len(videos_with_ar)}) Downloading video ID: {vid}")
        if download_video_and_subs(vid, DOWNLOAD_DIR):
            # Identify the audio and subtitle files
            audio_file = None
            subtitle_file = None

            # Check downloaded files
            for f in os.listdir(DOWNLOAD_DIR):
                if f.startswith(vid):
                    ext = os.path.splitext(f)[1].lower()
                    if ext in [".m4a", ".webm", ".opus", ".mp3", ".mp4"]:
                        audio_file = os.path.join(DOWNLOAD_DIR, f)
                    elif ext == ".vtt":
                        subtitle_file = os.path.join(DOWNLOAD_DIR, f)

            if audio_file and subtitle_file:
                # Convert to WAV
                wav_output = os.path.join(WAV_DIR, f"{vid}.wav")
                convert_to_wav(audio_file, wav_output)

                # Move the subtitle file to transcriptions folder
                new_subtitle_path = os.path.join(TRANSCRIPTIONS_DIR, f"{vid}.vtt")
                os.rename(subtitle_file, new_subtitle_path)

                # We no longer need the original audio file
                os.remove(audio_file)

                video_url = f"https://www.youtube.com/watch?v={vid}"
                metadata.append([vid, video_url, new_subtitle_path, wav_output])
            else:
                print(f"Warning: Could not find audio/subtitle for {vid}.")
                # Remove partial files if they exist
                if audio_file and os.path.exists(audio_file):
                    os.remove(audio_file)
                if subtitle_file and os.path.exists(subtitle_file):
                    os.remove(subtitle_file)
        else:
            print(f"Failed to download {vid}.")



if __name__ == "__main__":
    main()


Searching for videos...
Searching for query: أخبار عربية
Searching for query: أخبار مصرية
Found 500 videos from search.
Checking for Arabic transcripts...
Found 379 videos with Arabic transcripts.
(1/379) Downloading video ID: 5IjFyZsjLlA
(2/379) Downloading video ID: 9GuTIlv9l7A
(3/379) Downloading video ID: 8CQtiAzCUCs
(4/379) Downloading video ID: WJqZuIunV8k
(5/379) Downloading video ID: cgcJ_t-DCCU
(6/379) Downloading video ID: -1yEJgb5Tos
(7/379) Downloading video ID: rtiZKzMTxoE
(8/379) Downloading video ID: IL806qyjNZY
(9/379) Downloading video ID: 2oTQtiFLh9A
(10/379) Downloading video ID: ZDffZKnGxmw
(11/379) Downloading video ID: _-t6GKuurgM
(12/379) Downloading video ID: oh5QG85XAmg
(13/379) Downloading video ID: _TehyWd-i2s
(14/379) Downloading video ID: FS6NN0nr9SI
(15/379) Downloading video ID: LPnDuJ7Ctv4
(16/379) Downloading video ID: 8tLFroAkmz8
(17/379) Downloading video ID: M3p-SvhoWMA
(18/379) Downloading video ID: UMlsBsz_btM
(19/379) Downloading video ID: wM-k5K-

## Data preprocessing


### Renaming the wavs 

In [ ]:
import os
import shutil
from tqdm import tqdm

# ------------------- USER CONFIGURATION ------------------- #
WAV_DIR = "wavs"  # Directory containing the original WAV files
TRANSCRIPTIONS_DIR = "transcriptions"  # Directory containing the transcription files

OUTPUT_WAV_DIR = "wavs_renamed"  # Directory to store renamed WAV files
OUTPUT_TRANSCRIPTIONS_DIR = "transcriptions_renamed"  # Directory to store renamed transcription files

# Define the number of digits for numerical naming (e.g., 3 for '001')
NUM_DIGITS = 3

# ------------------- PREPARATION ------------------- #
# Create output directories if they don't exist
os.makedirs(OUTPUT_WAV_DIR, exist_ok=True)
os.makedirs(OUTPUT_TRANSCRIPTIONS_DIR, exist_ok=True)

# List all WAV files
wav_files = sorted([f for f in os.listdir(WAV_DIR) if f.lower().endswith('.wav')])

if not wav_files:
    print(f"No .wav files found in the directory: {WAV_DIR}")
    exit(1)

total_files = len(wav_files)
print(f"Total .wav files found: {total_files}")

# ------------------- RENAMING PROCESS ------------------- #
for idx, wav_file in enumerate(tqdm(wav_files, desc="Renaming Files"), start=1):
    # Create a new numerical name with leading zeros
    new_base_name = str(idx).zfill(NUM_DIGITS)

    # Define source and destination paths for WAV
    src_wav = os.path.join(WAV_DIR, wav_file)
    dst_wav = os.path.join(OUTPUT_WAV_DIR, f"{new_base_name}.wav")

    # Move and rename the WAV file
    shutil.copy(src_wav, dst_wav)  # Use shutil.move(src_wav, dst_wav) to move instead of copy

    # Define corresponding transcription filename
    base_name = os.path.splitext(wav_file)[0]
    trans_file = base_name + ".vtt"
    src_trans = os.path.join(TRANSCRIPTIONS_DIR, trans_file)

    if os.path.exists(src_trans):
        dst_trans = os.path.join(OUTPUT_TRANSCRIPTIONS_DIR, f"{new_base_name}.vtt")
        shutil.copy(src_trans, dst_trans)  # Use shutil.move(src_trans, dst_trans) to move
    else:
        print(f"Warning: Transcription file not found for {wav_file} -> {trans_file}")

print("Renaming complete!")
print(f"Renamed WAV files are in '{OUTPUT_WAV_DIR}/'")
print(f"Renamed transcription files are in '{OUTPUT_TRANSCRIPTIONS_DIR}/'")


Total .wav files found: 379


Renaming Files: 100%|██████████| 379/379 [01:21<00:00,  4.64it/s]

Renaming complete!
Renamed WAV files are in 'wavs_renamed/'
Renamed transcription files are in 'transcriptions_renamed/'


### Segmentation of wavs

In [ ]:
import os
from pydub import AudioSegment
import webvtt

# Configuration
WAV_DIR = "wavs_renamed"
TRANS_DIR = "transcriptions_renamed"
OUTPUT_DIR = "segments"
OUTPUT_TXT = "segments.txt"
MAX_SEG_DURATION = 15 * 1000  # 15 seconds in milliseconds

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# List all wav files (assuming they follow a pattern like 001.wav, 002.wav, etc.)
wav_files = [f for f in os.listdir(WAV_DIR) if f.lower().endswith(".wav")]
wav_files.sort()

with open(OUTPUT_TXT, "w", encoding="utf-8") as out_file:
    for wav_file in wav_files:
        base_name = os.path.splitext(wav_file)[0]
        wav_path = os.path.join(WAV_DIR, wav_file)
        vtt_path = os.path.join(TRANS_DIR, base_name + ".vtt")

        # Load the audio
        audio = AudioSegment.from_wav(wav_path)
        duration = len(audio)  # in milliseconds

        # Parse the VTT captions
        # webvtt gives us captions with start and end times in seconds
        if not os.path.exists(vtt_path):
            print(f"Warning: No VTT file found for {wav_file}")
            captions = []
        else:
            captions = list(webvtt.read(vtt_path))

        # We'll create segments of up to 15 seconds each
        # For a file of duration D ms, number of segments = ceil(D / MAX_SEG_DURATION)
        num_segments = (duration + MAX_SEG_DURATION - 1) // MAX_SEG_DURATION

        for i in range(num_segments):
            segment_start_ms = i * MAX_SEG_DURATION
            segment_end_ms = min((i+1)*MAX_SEG_DURATION, duration)

            # Extract audio segment
            segment_audio = audio[segment_start_ms:segment_end_ms]

            # Name the segment file, for example "001_0001.wav" if we want to keep track of original number
            # If you just want sequential naming like in the prompt, you can do that as well.
            # The prompt shows something like "0002.wav" - let's keep the original file base name and add segment index.
            # If you want just running numbers, you'd have to track globally. Here we will just do baseName_index.wav.
            segment_name = f"{base_name}_{i+1:04d}.wav"
            # If you want just "0001.wav", "0002.wav" continuously, you'd need a global counter.
            # But the prompt shows an example "0002.wav" which implies a certain naming pattern.
            # Let's assume we keep the original pattern: base name + index. You can adjust as needed.

            segment_path = os.path.join(OUTPUT_DIR, segment_name)
            segment_audio.export(segment_path, format="wav")

            # Now we find the transcripts that fall into this time range
            # captions have start and end in seconds, convert to ms
            segment_start_sec = segment_start_ms / 1000.0
            segment_end_sec = segment_end_ms / 1000.0

            segment_texts = []
            for caption in captions:
                # caption.start and caption.end are strings like "00:00:10.000"
                # webvtt captions have start_in_seconds and end_in_seconds properties
                if caption.start_in_seconds < segment_end_sec and caption.end_in_seconds > segment_start_sec:
                    # The caption overlaps this segment
                    segment_texts.append(caption.text.strip())

            # Join all text lines for this segment
            # If you want all text concatenated on one line:
            transcript_line = " ".join(segment_texts)

            # Write to the output text file
            # Format: "ARA NORM  {filename}.wav" "{transcript}"
            out_file.write(f"ARA NORM  {segment_name} \"{transcript_line}\"\n")

print("Splitting complete. Segments and transcript mapping are ready!")


### Preprocessing audio

In [ ]:
import os
import torch
import torchaudio
import librosa
import numpy as np
from tqdm import tqdm

# Modify these paths as needed
SEGMENTS_DIR = 'segments'           # Source directory with segment .wav files
SEGMENTS_PROCESSED_DIR = 'segments_processed'  # Destination directory for processed files

sr_target = 22050
silence_audio_size = 256 * 3
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Get all .wav files from the segments directory
wav_fpaths = [f.path for f in os.scandir(SEGMENTS_DIR) if f.path.endswith('.wav')]

# Create output directory if it doesn't exist
if not os.path.exists(SEGMENTS_PROCESSED_DIR):
    os.makedirs(SEGMENTS_PROCESSED_DIR)
    print(f"Created folder @ {SEGMENTS_PROCESSED_DIR}")

for wav_fpath in tqdm(wav_fpaths, desc="Processing"):
    fname = os.path.basename(wav_fpath)
    fpath = os.path.join(SEGMENTS_DIR, fname)

    # Load audio
    wave, sr = torchaudio.load(fpath)

    # Resample if needed
    if sr != sr_target:
        wave = wave.to(device)
        wave = torchaudio.functional.resample(
            wave, sr, sr_target, lowpass_filter_width=1024
        )
    
    # Convert to numpy and normalize
    wave_ = wave[0].cpu().numpy()
    max_val = np.abs(wave_).max()
    if max_val > 0:
        wave_ = wave_ / max_val * 0.999

    # Trim silence from the start and end
    wave_, _ = librosa.effects.trim(
        wave_, top_db=23, frame_length=1024, hop_length=256
    )

    # Append silence at the end
    wave_ = np.append(wave_, [0.0] * silence_audio_size)

    # Save processed file
    out_path = os.path.join(SEGMENTS_PROCESSED_DIR, fname)
    torchaudio.save(out_path, torch.tensor(wave_).unsqueeze(0), sr_target)

print("Processing complete!")
print(f"Processed files are in '{SEGMENTS_PROCESSED_DIR}'")


### Transcription using whisper

In [ ]:
pip install git+https://github.com/openai/whisper.git


  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-a_s8dxjf
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-a_s8dxjf
  Resolved https://github.com/openai/whisper.git to commit 90db0de1896c23cbfaf0c58bc2d30665f709f170
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.5/209.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.5 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20240930-py3-none-any.whl size=803583 sha256=c3218ccfb7bf69d953ffa217d09a8d64338b442052e47dfc5abfc8fb7c75d588
  Stored in directory: /tmp/pip-ephem-wheel-cache-o46_zz2o/wheels/8b/6c/d0/622666868c179f156cf595c8b6f06f88bc5d80c4b31dccaa03
Successfully built openai-whisper


In [ ]:
import os
import whisper

In [ ]:
# ---------------- USER CONFIGURATION ----------------
WAV_DIR = "/content/drive/MyDrive/my_dataset/segments_processed"    # Directory containing your .wav files
TRANSCRIPT_DIR = "/content/drive/MyDrive/my_dataset/transcriptions" # Directory for .txt files
MODEL_SIZE = "medium"  # Choose from: tiny, base, small, medium, large
LANGUAGE = "ar"         # Set to "ar" for Arabic if desired; None for auto detection
# ----------------------------------------------------

# Create the transcript directory if it doesn't exist
os.makedirs(TRANSCRIPT_DIR, exist_ok=True)

# Load the Whisper model
model = whisper.load_model(MODEL_SIZE)

# List all wav files
wav_files = [f for f in os.listdir(WAV_DIR) if f.lower().endswith('.wav')]

for wav_file in wav_files:
    wav_path = os.path.join(WAV_DIR, wav_file)
    print(f"Transcribing: {wav_file} ...")

    # Transcribe the audio
    transcription = model.transcribe(wav_path, language=LANGUAGE, task="transcribe")
    text = transcription['text'].strip()

    # Print transcript for reference
    print(f"Transcript for {wav_file}: {text}")

    # Write result to a .txt file in the transcripts directory
    base_name = os.path.splitext(wav_file)[0]
    txt_path = os.path.join(TRANSCRIPT_DIR, base_name + ".txt")

    with open(txt_path, "w", encoding="utf-8") as txt_file:
        txt_file.write(text + "\n")

print("All transcriptions completed!")


100%|█████████████████████████████████████| 1.42G/1.42G [00:17<00:00, 87.3MiB/s]
/usr/local/lib/python3.10/dist-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this exper

Streaming output truncated to the last 5000 lines.
Transcript for 275_0032.wav: لم ترد ان بقى حتى الان عن وقوى خسائر بشري او مدية جراء هذا الزلزال ننتقل الى خبر اخر
Transcribing: 006_0004.wav ...
Transcript for 006_0004.wav: اهلا بكم مشاهدين الاخارم هذه نشرة الأخبار تأتيكم من قناة هيئة
Transcribing: 065_0034.wav ...
Transcript for 065_0034.wav: تدوير المواد الغذائية التي ستكون الأولى في شرق إفريقيا وبالطبع نحن نعلم كيف ترتبط إيثيوبيا جيدا بميناء جيبوتي وجميع الخدمات اللوجستية
Transcribing: 310_0040.wav ...
Transcript for 310_0040.wav: منذ عام الف وتسعمائة وواحد وثمانين بلا محاكمة. اما اليوم وبعد كل هذه العقود اطلق سراح التطري مع فتح السجون وسقوط نظام بشر الاسد.
Transcribing: 182_0011.wav ...
Transcript for 182_0011.wav: عقد التيثيوبيا واليابان مشاوراتهم السياسية الافتتاحية يوم أمس في العاصمة اليابانية توكيو وترأس الاجتماع السفير مسجان ورقة وزيودي
Transcribing: 242_0022.wav ...
Transcript for 242_0022.wav: الالتزام الولايات المتحدة الأمريكية المستمر بالعمل مع إثيوبيا لمكافحة الارهاب وال

### Splitting data and conversion to phonomes

In [4]:
import os
import csv
import random
import subprocess
from tqdm import tqdm

# Import necessary functions from your provided scripts
from text.__init__ import arabic_to_buckwalter  

In [5]:
# ---------------- USER CONFIGURATION ----------------
DATA_DIR = "/content/drive/MyDrive/my_dataset"  # Base directory
WAV_DIR = os.path.join(DATA_DIR, "segments_processed")      # Directory containing .wav files
TRANSCRIPT_DIR = os.path.join(DATA_DIR, "transcriptions")  # Directory containing .txt transcription files
PHONETISE_SCRIPT = os.path.join("/content/text", "phonetise_buckwalter.py")  # Path to phonetise_buckwalter.py

In [6]:
# Split ratio for train/test
TRAIN_RATIO = 0.9

# Paths for output
TRAIN_ARAB = os.path.join(DATA_DIR, "train_arab.txt")
TEST_ARAB = os.path.join(DATA_DIR, "test_arab.txt")
TRAIN_BUCKW = os.path.join(DATA_DIR, "train_buckw.txt")
TEST_BUCKW = os.path.join(DATA_DIR, "test_buckw.txt")
TRAIN_PHON = os.path.join(DATA_DIR, "train_phon.txt")
TEST_PHON = os.path.join(DATA_DIR, "test_phon.txt")

# ----------------------------------------------------


In [10]:
# Step 1: Collect (wav_file, transcription)
print("Collecting wav and transcription pairs...")
entries = []

for f in os.listdir(WAV_DIR):
    if f.lower().endswith(".wav"):
        base = os.path.splitext(f)[0]
        wav_path = os.path.join(WAV_DIR, f)
        txt_path = os.path.join(TRANSCRIPT_DIR, base + ".txt")
        if os.path.exists(txt_path):
            with open(txt_path, 'r', encoding='utf-8') as tf:
                text = tf.read().strip()
            entries.append((base, text))
            print(f"Successfully processed: {f}")  # Print statement for each successful file
        else:
            print(f"Warning: No transcription found for {f}, skipping.")

print(f"Total entries collected: {len(entries)}")


Streaming output truncated to the last 5000 lines.
Successfully processed: 009_0005.wav
Successfully processed: 252_0028.wav
Successfully processed: 197_0019.wav
Successfully processed: 229_0040.wav
Successfully processed: 191_0017.wav
Successfully processed: 159_0005.wav
Successfully processed: 196_0037.wav
Successfully processed: 081_0001.wav
Successfully processed: 154_0009.wav
Successfully processed: 173_0018.wav
Successfully processed: 375_0021.wav
Successfully processed: 252_0026.wav
Successfully processed: 372_0047.wav
Successfully processed: 351_0020.wav
Successfully processed: 076_0046.wav
Successfully processed: 200_0008.wav
Successfully processed: 133_0038.wav
Successfully processed: 224_0020.wav
Successfully processed: 186_0034.wav
Successfully processed: 205_0018.wav
Successfully processed: 379_0061.wav
Successfully processed: 180_0036.wav
Successfully processed: 231_0048.wav
Successfully processed: 066_0039.wav
Successfully processed: 203_0035.wav
Successfully processed: 

In [11]:
# Step 2: Split into train and test
print("Splitting data into train and test sets...")
random.shuffle(entries)
train_count = int(len(entries) * TRAIN_RATIO)
train_entries = entries[:train_count]
test_entries = entries[train_count:]

# Function to write entries in the desired format
def write_list(file_path, entries, mode="arab"):
    with open(file_path, 'w', encoding='utf-8') as f:
        for base, text in entries:
            wav_filename = f"{base}.wav"
            if mode == "arab":
                # Write in the format: "0002.wav" "Arabic text"
                f.write(f"\"{wav_filename}\" \"{text}\"\n")
            elif mode == "buckw":
                # Write in the format: "0002.wav" "Buckwalter transliteration"
                buckwalter_text = arabic_to_buckwalter(text)
                f.write(f"\"{wav_filename}\" \"{buckwalter_text}\"\n")
            else:
                raise ValueError("Unsupported mode. Choose 'arab' or 'buckw'.")

# Step 3: Write train_arab.txt and test_arab.txt
print("Writing train_arab.txt and test_arab.txt...")
write_list(TRAIN_ARAB, train_entries, mode="arab")
write_list(TEST_ARAB, test_entries, mode="arab")
print(f"Training Arabic list saved to {TRAIN_ARAB}")
print(f"Testing Arabic list saved to {TEST_ARAB}")


Splitting data into train and test sets...
Writing train_arab.txt and test_arab.txt...
Training Arabic list saved to /content/drive/MyDrive/my_dataset/train_arab.txt
Testing Arabic list saved to /content/drive/MyDrive/my_dataset/test_arab.txt


In [12]:
# Step 4: Write train_buckw.txt and test_buckw.txt
print("Writing train_buckw.txt and test_buckw.txt...")
write_list(TRAIN_BUCKW, train_entries, mode="buckw")
write_list(TEST_BUCKW, test_entries, mode="buckw")
print(f"Training Buckwalter list saved to {TRAIN_BUCKW}")
print(f"Testing Buckwalter list saved to {TEST_BUCKW}")



Writing train_buckw.txt and test_buckw.txt...
Training Buckwalter list saved to /content/drive/MyDrive/my_dataset/train_buckw.txt
Testing Buckwalter list saved to /content/drive/MyDrive/my_dataset/test_buckw.txt


In [13]:
# Step 5: Generate phoneme sequences directly
print("Converting Buckwalter transliterations to phoneme sequences...")

from text.__init__ import buckwalter_to_phonemes  # Assuming your Buckwalter to phoneme conversion function is here

def generate_phoneme_txt(wav_list_path, phon_txt_path, mode="training"):
    with open(wav_list_path, 'r', encoding='utf-8') as wl, \
         open(phon_txt_path, 'w', encoding='utf-8') as ptf:
        wav_lines = wl.readlines()
        for wav_line in wav_lines:
            # Extract WAV filename and Buckwalter text
            wav_filename, buckwalter_text = map(str.strip, wav_line.strip().split('"')[1::2])
            try:
                # Convert Buckwalter to phonemes directly
                phoneme_sequence = buckwalter_to_phonemes(buckwalter_text)
                # Write the result in the format: "filename" "phoneme_sequence"
                ptf.write(f"\"{wav_filename}\" \"{phoneme_sequence}\"\n")
            except Exception as e:
                print(f"Error processing {wav_filename}: {e}")
                continue

# Generate phoneme text files directly
generate_phoneme_txt(TRAIN_BUCKW, TRAIN_PHON, mode="training")
generate_phoneme_txt(TEST_BUCKW, TEST_PHON, mode="testing")

print(f"Phoneme sequences for training saved to {TRAIN_PHON}")
print(f"Phoneme sequences for testing saved to {TEST_PHON}")


Converting Buckwalter transliterations to phoneme sequences...
Phoneme sequences for training saved to /content/drive/MyDrive/my_dataset/train_phon.txt
Phoneme sequences for testing saved to /content/drive/MyDrive/my_dataset/test_phon.txt


## Training Model

In [ ]:
!git clone https://github.com/nipponjo/tts-arabic-pytorch.git

!cp -r /content/tts-arabic-pytorch /content/drive/MyDrive/


In [ ]:
%cd /content/drive/MyDrive/tts-arabic-pytorch
!python train.py --config './configs/nawar_tc2_adv.yaml'


/content/drive/MyDrive/tts-arabic-pytorch
2024-12-21 14:54:51.580352: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-21 14:54:51.598336: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-21 14:54:51.619353: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-21 14:54:51.625697: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-21 14:54:51.640961: I